# Retrain Student Risk Model

This notebook retrains the student-risk XGBoost model using the same 6 features expected by the backend.

Backend feature order:
- `previous_gpa`
- `failed_subject_count`
- `attendance_rate`
- `academic_challenge_score`
- `external_factor_score`
- `received_academic_support`


## Setup

Install these once in your VS Code Python environment if needed:

```python
!pip install pandas scikit-learn xgboost openpyxl
```

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import pickle

PROJECT_ROOT = Path.cwd()
BACKEND_DIR = PROJECT_ROOT
if (PROJECT_ROOT / 'backend').exists():
    BACKEND_DIR = PROJECT_ROOT / 'backend'

BACKEND_DIR

In [ ]:
# Set your dataset path here.
# Supported examples: .csv, .xlsx
DATASET_PATH = BACKEND_DIR / 'uploads' / 'student_risk_training_data.xlsx'
DATASET_PATH

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

if DATASET_PATH.suffix.lower() == '.csv':
    df = pd.read_csv(DATASET_PATH)
else:
    df = pd.read_excel(DATASET_PATH)

df.head()

## Configure column names

Change these only if your dataset uses different headers.

In [ ]:
FEATURE_COLUMNS = [
    'previous_gpa',
    'failed_subject_count',
    'attendance_rate',
    'academic_challenge_score',
    'external_factor_score',
    'received_academic_support',
]

# Update this if your target column has a different name.
TARGET_COLUMN = 'risk_label'

FEATURE_COLUMNS, TARGET_COLUMN

In [ ]:
missing = [col for col in FEATURE_COLUMNS + [TARGET_COLUMN] if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

training_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
training_df = training_df.dropna()

# Normalize binary target values into 0/1.
target_map = {
    'high': 1,
    'low': 0,
    'at_risk': 1,
    'not_at_risk': 0,
    'yes': 1,
    'no': 0,
}

def normalize_target(value):
    if isinstance(value, str):
        key = value.strip().lower()
        if key in target_map:
            return target_map[key]
    if value in (0, 1):
        return int(value)
    raise ValueError(f'Unsupported target value: {value!r}')

training_df[TARGET_COLUMN] = training_df[TARGET_COLUMN].apply(normalize_target)
training_df.head()

In [ ]:
X = training_df[FEATURE_COLUMNS].copy()
y = training_df[TARGET_COLUMN].copy()

X['failed_subject_count'] = X['failed_subject_count'].astype(int)
X['received_academic_support'] = X['received_academic_support'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    random_state=42,
)

model.fit(X_train, y_train)
model

In [ ]:
preds = model.predict(X_test)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, digits=4))

In [ ]:
importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

importance

## Save the model

This saves both formats:
- `.json` for safer XGBoost loading in production
- `.pkl` for compatibility with the current app


In [ ]:
json_path = BACKEND_DIR / 'xgboost_student_risk.json'
pkl_path = BACKEND_DIR / 'xgboost_student_risk.pkl'

model.save_model(json_path)
with pkl_path.open('wb') as f:
    pickle.dump(model, f)

print('Saved JSON model to:', json_path)
print('Saved PKL model to:', pkl_path)

In [ ]:
# Quick smoke test prediction using one row from X_test.
sample_features = X_test.iloc[[0]]
sample_prediction = model.predict(sample_features)[0]
sample_probability = model.predict_proba(sample_features)[0][1]

print('Sample features:')
print(sample_features)
print('Prediction:', int(sample_prediction))
print('High-risk probability:', round(float(sample_probability) * 100, 2), '%')